# 🚀 SABER Training & Evaluation on Google Colab (T4 GPU)

This notebook handles setting up, training, and evaluating the **SABER** (Sensor-Agnostic Bridged Embedding Retrieval) framework on a Google Colab T4 GPU runtime.

### ⚠️ IMPORTANT: Sync Local Changes Before Running
Before running this notebook, ensure you have committed and pushed your latest local changes to GitHub:
```bash
git add .
git commit -m "Implement Round 2 modifications: 3-layer Projection Head + expanded LoRA + gradient flow fix"
git push origin master
```

---

## 📂 Step 1: Mount Google Drive
We mount Google Drive to access the shared `SABER_Data` folder shortcut on your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 🛠️ Step 2: Clone Repository, Install Dependencies & Pre-Download Weights

In [ ]:
import os

# Clone or pull the repository to fetch the latest changes
if os.path.exists("/content/SABER"):
    print("SABER directory already exists. Pulling the latest changes...")
    %cd /content/SABER
    !git pull
else:
    print("Cloning SABER repository...")
    !git clone https://github.com/SK8-infi/SABER.git
    %cd SABER

# Uninstall incompatible torchao to prevent PEFT ImportError in Google Colab
!pip uninstall -y torchao

# Install required packages
!pip install -r Saber/requirements.txt
!pip install albumentations --upgrade

# Set up paths for DOFA model weights
!mkdir -p /root/.cache/torch/hub/checkpoints/

drive_weights = "/content/drive/MyDrive/SABER_Data/DOFA_ViT_base_e100.pth"
local_weights = "/root/.cache/torch/hub/checkpoints/DOFA_ViT_base_e100.pth"

# Clean up any previously corrupted file (e.g. error HTML page under 1MB)
if os.path.exists(local_weights) and os.path.getsize(local_weights) < 1000000:
    print("Cleaning up corrupted local cache file...")
    os.remove(local_weights)
if os.path.exists(drive_weights) and os.path.getsize(drive_weights) < 1000000:
    print("Cleaning up corrupted weights file on Google Drive...")
    os.remove(drive_weights)

# Handle loading/caching
if os.path.exists(drive_weights):
    print("Found DOFA weights on Google Drive. Copying to local cache (fast)... ")
    !cp "{drive_weights}" "{local_weights}"
else:
    print("DOFA weights not found on Google Drive. Downloading from HF...")
    !huggingface-cli download earthflow/DOFA DOFA_ViT_base_e100.pth --local-dir /root/.cache/torch/hub/checkpoints/ --local-dir-use-symlinks False
    print("Backing up weights to Google Drive for future runs...")
    !cp "{local_weights}" "{drive_weights}"

# Define log-syncing function to append local logs to Google Drive
def sync_logs():
    import os
    local_log_path = "logs/saber.log"
    drive_log_dir = "/content/drive/MyDrive/SABER_Data/logs"
    os.makedirs(drive_log_dir, exist_ok=True)
    drive_log_path = os.path.join(drive_log_dir, "saber_colab.log")
    
    if os.path.exists(local_log_path):
        with open(local_log_path, "r", encoding="utf-8") as f_local:
            local_content = f_local.read()
        
        if local_content.strip():
            # Append logs to the Google Drive log file
            with open(drive_log_path, "a", encoding="utf-8") as f_drive:
                f_drive.write("\n" + "="*60 + "\n")
                f_drive.write(f"--- LOG SYNC: {local_log_path} ---\n")
                f_drive.write("="*60 + "\n")
                f_drive.write(local_content)
            
            # Clear local log file to prevent duplicate appends later
            with open(local_log_path, "w", encoding="utf-8") as f_local:
                f_local.write("")
            print(f"Successfully appended local logs to Google Drive: {drive_log_path}")
        else:
            print("Local log file is empty. Nothing to sync.")
    else:
        print("No local log file found to sync.")

## 📦 Step 3: Link or Extract Datasets from Shared Google Drive Shortcut
Prepare the datasets. If archive files (`benv1_14k.tgz`/`benv1_14k.zip` or `DSRSID-001.zip`) are found in the shared drive, they will be extracted locally (which is extremely fast and avoids browser crashes during upload). Otherwise, it falls back to symlinking raw folders.

In [ ]:
import os

# Create local Datasets directories
!mkdir -p Datasets/DSRSID

drive_saber_data = "/content/drive/MyDrive/SABER_Data"

# 1. Handle DSRSID Dataset (Check for DSRSID-001.zip)
# dsrsid_zip = os.path.join(drive_saber_data, "DSRSID-001.zip")
# if os.path.exists(dsrsid_zip):
#     print("Found DSRSID-001.zip on Google Drive. Unzipping locally (fast)...")
#     !unzip -q "{dsrsid_zip}" -d Datasets/DSRSID/
#     # Move file if it unzipped into a nested folder
#     if os.path.exists("Datasets/DSRSID/DSRSID/DSRSID-001.mat"):
#         !mv Datasets/DSRSID/DSRSID/* Datasets/DSRSID/
#         !rmdir Datasets/DSRSID/DSRSID
# else:
#     # Fallback to linking if they uploaded the raw file
#     raw_mat = os.path.join(drive_saber_data, "DSRSID/DSRSID-001.mat")
#     if os.path.exists(raw_mat):
#         print("Linking raw DSRSID mat file...")
#         !ln -s "{raw_mat}" Datasets/DSRSID/DSRSID-001.mat
#     else:
#         # Check if the mat file was placed directly in SABER_Data
#         raw_mat_alt = os.path.join(drive_saber_data, "DSRSID-001.mat")
#         if os.path.exists(raw_mat_alt):
#             print("Linking raw DSRSID mat file (root)...")
#             !ln -s "{raw_mat_alt}" Datasets/DSRSID/DSRSID-001.mat
#         else:
#             print("⚠️ DSRSID dataset not found on Google Drive!")

# 2. Handle BEN-14K Dataset (Check for benv1_14k.tgz first, then benv1_14k.zip)
ben_tgz = os.path.join(drive_saber_data, "benv1_14k.tgz")
ben_zip = os.path.join(drive_saber_data, "benv1_14k.zip")

if os.path.exists(ben_tgz):
    print("Found benv1_14k.tgz on Google Drive. Extracting locally (fast)...")
    !tar -xf "{ben_tgz}" -C Datasets/
    # Move files if they extracted into a nested folder
    if os.path.exists("Datasets/benv1_14k/benv1_14k"):
        !mv Datasets/benv1_14k/benv1_14k/* Datasets/benv1_14k/
        !rmdir Datasets/benv1_14k/benv1_14k
elif os.path.exists(ben_zip):
    print("Found benv1_14k.zip on Google Drive. Unzipping locally (fast)...")
    !unzip -q "{ben_zip}" -d Datasets/
    # Move files if they unzipped into a nested folder
    if os.path.exists("Datasets/benv1_14k/benv1_14k"):
        !mv Datasets/benv1_14k/benv1_14k/* Datasets/benv1_14k/
        !rmdir Datasets/benv1_14k/benv1_14k
else:
    raw_ben = os.path.join(drive_saber_data, "benv1_14k")
    if os.path.exists(raw_ben):
        print("Linking raw benv1_14k folder...")
        !ln -s "{raw_ben}" Datasets/benv1_14k
    else:
        print("⚠️ BEN-14K dataset not found on Google Drive!")

# Verification check
print("\n--- Local Datasets Structure ---")
!ls -lh Datasets/DSRSID/
!ls -lh Datasets/

## ⚡ Step 4: Run Training & Feature Extraction (Round 2)

Run training for **BEN-14K** and/or **DSRSID** using the updated Round 2 configuration (3-layer Projection Head + expanded LoRA + fixed gradient flow).

In [ ]:
# ==========================================================
# PIPELINE A: BEN-14K
# ==========================================================

# 1. Train the Encoder (20 epochs for faster iteration)
!python Saber/train.py --dataset_name ben14k --modality both --data_dir Datasets/benv1_14k --epochs 20 --synthetic false

# Copy encoder checkpoint to prevent overwriting
!cp checkpoints/latest.pth checkpoints/latest_ben14k.pth

# 2. Extract features
!python Saber/extract_features.py --dataset_name ben14k --data_dir Datasets/benv1_14k --synthetic false --checkpoint checkpoints/latest_ben14k.pth --output_dir checkpoints/extracted_ben14k

# 3. Train the Bridge (80 epochs for faster iteration)
!python Saber/train_bridge.py --features_dir checkpoints/extracted_ben14k --epochs 80 --ode_steps 10

# Copy bridge checkpoint
!cp checkpoints/bridge_best.pth checkpoints/bridge_best_ben14k.pth

# Sync training logs to Google Drive (extended/appended)
sync_logs()

## 📊 Step 5: Evaluate Models

In [ ]:
# 1. Evaluate BEN-14K Same-Modal retrieval (S2 -> S2)
!python Saber/evaluate.py --architecture saber --dataset_name ben14k --modality s2 --synthetic false --data_dir Datasets/benv1_14k --checkpoint checkpoints/latest_ben14k.pth

# 2. Evaluate BEN-14K Cross-Modal retrieval (S1 -> S2)
!cp checkpoints/bridge_best_ben14k.pth checkpoints/bridge_best.pth
!python Saber/evaluate.py --architecture saber --dataset_name ben14k --modality both --synthetic false --data_dir Datasets/benv1_14k --checkpoint checkpoints/latest_ben14k.pth

# Sync evaluation logs to Google Drive (extended/appended)
sync_logs()

## 💾 Step 6: Backup Checkpoints to Shared Google Drive Folder
Persist checkpoints back to the shared `SABER_Data/checkpoints` folder so they are saved permanently for all team members.

In [ ]:
# Create checkpoints backup directory inside the shared Drive folder
!mkdir -p "/content/drive/MyDrive/SABER_Data/checkpoints"

# Copy encoder and bridge checkpoints to the shared drive
!cp checkpoints/latest_ben14k.pth "/content/drive/MyDrive/SABER_Data/checkpoints/"
!cp checkpoints/bridge_best_ben14k.pth "/content/drive/MyDrive/SABER_Data/checkpoints/"
# !cp checkpoints/latest_dsrsid.pth "/content/drive/MyDrive/SABER_Data/checkpoints/"
# !cp checkpoints/bridge_best_dsrsid.pth "/content/drive/MyDrive/SABER_Data/checkpoints/"

print("Checkpoints successfully backed up to shared Drive folder!")